# 🫀 Pandas Assignment: Data Cleaning, EDA, and Business Insights
## Healthcare Analytics — Heart Disease Dataset

**Dataset Source:** [Heart Disease Dataset (Kaggle)](https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset)

**Objective:** This notebook demonstrates how to use Pandas for practical healthcare data analysis. By the end, we will be able to:
- Load datasets using Pandas
- Inspect dataset structure
- Identify missing values, duplicates, and invalid records
- Clean and transform raw data
- Perform filtering, sorting, grouping, and aggregation
- Create meaningful visualizations
- Generate business/analytical insights
- Export cleaned datasets and summary reports

> 💡 An optional Kaggle API download cell is included if you'd prefer to pull the dataset directly from Kaggle instead of the mirrored CSV link used here.

## Step 1: Import Libraries
We import `pandas` for data handling, `numpy` for numerical operations, and `matplotlib`/`seaborn` for visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

## Step 2 (Optional): Download Dataset via Kaggle API
Skip this if using the direct CSV link in Step 3 below.

In [ ]:
# Uncomment to fetch via Kaggle API instead

# !pip install kaggle
# from google.colab import files
# files.upload()  # upload your kaggle.json here
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d johnsmith88/heart-disease-dataset
# !unzip heart-disease-dataset.zip

## Step 3: Load the Dataset
We use `pd.read_csv()` to load the Heart Disease dataset into a pandas DataFrame.

**Column reference:**
- `age` — age in years
- `sex` — 1 = male, 0 = female
- `cp` — chest pain type (0–3)
- `trestbps` — resting blood pressure (mm Hg)
- `chol` — serum cholesterol (mg/dl)
- `fbs` — fasting blood sugar > 120 mg/dl (1 = true, 0 = false)
- `restecg` — resting ECG results (0–2)
- `thalach` — maximum heart rate achieved
- `exang` — exercise-induced angina (1 = yes, 0 = no)
- `oldpeak` — ST depression induced by exercise
- `slope` — slope of peak exercise ST segment
- `ca` — number of major vessels colored by fluoroscopy (0–4)
- `thal` — thalassemia (0–3)
- `target` — 1 = heart disease present, 0 = no heart disease

In [ ]:
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/heart.csv'
df = pd.read_csv(url)

print('Dataset loaded successfully!')
print('Shape (rows, columns):', df.shape)

## Step 4: Initial Exploration
We inspect the dataset's structure: column names, data types, and a quick preview of records.

In [ ]:
# Preview first 5 rows
df.head()

In [ ]:
# Preview last 5 rows
df.tail()

In [ ]:
# Column names, data types, and non-null counts
df.info()

In [ ]:
# Statistical summary for numeric columns
df.describe()

In [ ]:
# List of all column names
df.columns.tolist()

## Step 5: Identify Missing Values, Duplicates, and Invalid Records
Before cleaning, we must first detect problems in the data.

In [ ]:
# Count missing values per column
df.isnull().sum()

In [ ]:
# Check for duplicate rows
print('Duplicate rows found:', df.duplicated().sum())

In [ ]:
# Check for invalid/out-of-range values (e.g. resting BP or cholesterol of 0 is physiologically impossible)
print('Rows with trestbps == 0:', (df['trestbps'] == 0).sum())
print('Rows with chol == 0:', (df['chol'] == 0).sum())
print('Rows with age <= 0:', (df['age'] <= 0).sum())

## Step 6: Data Cleaning
We handle the issues found above:
- Remove duplicate rows
- Replace invalid `0` values in `chol`/`trestbps` (impossible readings) with the column median
- Confirm data types are appropriate

In [ ]:
# Work on a copy so the original df is preserved for comparison
df_clean = df.copy()

# Drop exact duplicate rows
df_clean = df_clean.drop_duplicates()

# Replace physiologically invalid 0 values with the column median
for col in ['trestbps', 'chol']:
    median_val = df_clean.loc[df_clean[col] > 0, col].median()
    df_clean[col] = df_clean[col].replace(0, median_val)

print('Shape after cleaning:', df_clean.shape)
df_clean.isnull().sum()

## Step 7: Data Transformation
We create human-readable labels for some coded columns and bucket continuous variables — this makes grouping and visualization more intuitive.

In [ ]:
# Map coded columns to readable labels
df_clean['sex_label'] = df_clean['sex'].map({1: 'Male', 0: 'Female'})
df_clean['target_label'] = df_clean['target'].map({1: 'Disease', 0: 'No Disease'})

# Create age groups
bins = [0, 40, 50, 60, 70, 100]
labels = ['<40', '40-49', '50-59', '60-69', '70+']
df_clean['AgeGroup'] = pd.cut(df_clean['age'], bins=bins, labels=labels)

df_clean[['age', 'AgeGroup', 'sex_label', 'target_label']].head()

## Step 8: Filtering & Selecting Data
Pandas lets us slice data using column selection and conditional filters.

In [ ]:
# Select a few relevant columns
df_clean[['age', 'sex_label', 'chol', 'target_label']].head()

In [ ]:
# Filter: patients over 60 with heart disease
senior_patients = df_clean[(df_clean['age'] > 60) & (df_clean['target'] == 1)]
senior_patients[['age', 'sex_label', 'chol', 'trestbps']].head()

In [ ]:
# Filter: high cholesterol patients (chol > 300)
high_chol = df_clean[df_clean['chol'] > 300]
print('Number of high-cholesterol patients:', len(high_chol))
high_chol[['age', 'sex_label', 'chol', 'target_label']].head()

## Step 9: Sorting Data
`sort_values()` arranges rows by one or more columns.

In [ ]:
# Sort patients by cholesterol level, highest first
df_clean.sort_values('chol', ascending=False)[['age', 'sex_label', 'chol', 'target_label']].head(10)

## Step 10: Grouping & Aggregation
`groupby()` summarizes data by category — key for spotting patterns in healthcare data, e.g. disease rate by age group or gender.

In [ ]:
# Heart disease rate by gender
df_clean.groupby('sex_label')['target'].mean()

In [ ]:
# Heart disease rate by age group
df_clean.groupby('AgeGroup')['target'].mean()

In [ ]:
# Multiple aggregations: patient count, avg cholesterol, avg resting BP, disease rate — by age group and gender
df_clean.groupby(['AgeGroup', 'sex_label']).agg(
    PatientCount=('age', 'count'),
    AvgCholesterol=('chol', 'mean'),
    AvgRestingBP=('trestbps', 'mean'),
    DiseaseRate=('target', 'mean')
)

## Step 11: Visualizations
Charts make patterns in patient data easier to spot than raw tables.

In [ ]:
# Distribution of heart disease cases
plt.figure(figsize=(6,4))
sns.countplot(data=df_clean, x='target_label', palette='Set2')
plt.title('Heart Disease Distribution')
plt.xlabel('')
plt.show()

In [ ]:
# Disease rate by age group
plt.figure(figsize=(7,4))
sns.barplot(data=df_clean, x='AgeGroup', y='target', palette='coolwarm')
plt.title('Heart Disease Rate by Age Group')
plt.ylabel('Disease Rate')
plt.show()

In [ ]:
# Disease rate by gender
plt.figure(figsize=(6,4))
sns.barplot(data=df_clean, x='sex_label', y='target', palette='Set1')
plt.title('Heart Disease Rate by Gender')
plt.ylabel('Disease Rate')
plt.show()

In [ ]:
# Cholesterol distribution
plt.figure(figsize=(7,4))
sns.histplot(df_clean['chol'], bins=30, kde=True, color='darkred')
plt.title('Cholesterol Level Distribution')
plt.xlabel('Cholesterol (mg/dl)')
plt.show()

In [ ]:
# Age vs Max Heart Rate, colored by disease status
plt.figure(figsize=(7,5))
sns.scatterplot(data=df_clean, x='age', y='thalach', hue='target_label', alpha=0.7)
plt.title('Age vs Maximum Heart Rate Achieved')
plt.xlabel('Age')
plt.ylabel('Max Heart Rate')
plt.show()

In [ ]:
# Correlation heatmap (numeric columns only)
plt.figure(figsize=(9,7))
numeric_df = df_clean.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

## Step 12: Business / Analytical Insights
Translating the numbers into a written summary helps communicate findings to non-technical stakeholders (e.g. hospital administrators).

In [ ]:
overall_rate = df_clean['target'].mean() * 100
male_rate = df_clean[df_clean['sex_label']=='Male']['target'].mean() * 100
female_rate = df_clean[df_clean['sex_label']=='Female']['target'].mean() * 100
highest_risk_age_group = df_clean.groupby('AgeGroup')['target'].mean().idxmax()
avg_chol_disease = df_clean[df_clean['target']==1]['chol'].mean()
avg_chol_no_disease = df_clean[df_clean['target']==0]['chol'].mean()

print(f"Overall heart disease rate: {overall_rate:.1f}%")
print(f"Male disease rate: {male_rate:.1f}%")
print(f"Female disease rate: {female_rate:.1f}%")
print(f"Age group with highest disease rate: {highest_risk_age_group}")
print(f"Avg cholesterol (disease group): {avg_chol_disease:.1f} mg/dl")
print(f"Avg cholesterol (no disease group): {avg_chol_no_disease:.1f} mg/dl")

**Key Insights:**
- Heart disease prevalence and its variation by gender and age group are quantified above — use these printed rates to identify the highest-risk segments.
- Patients with heart disease tend to show different average cholesterol levels compared to those without — flagged above for further investigation.
- The correlation heatmap highlights which clinical variables (e.g. max heart rate, ST depression, chest pain type) are most associated with the disease outcome.
- These insights could help hospitals prioritize screening resources for higher-risk age/gender segments.

## Step 13: Export Cleaned Dataset and Summary Report

In [ ]:
# Export cleaned dataset
df_clean.to_csv('heart_disease_cleaned.csv', index=False)
print('Cleaned dataset saved as heart_disease_cleaned.csv')

In [ ]:
# Export a summary report (grouped statistics) as a separate CSV
summary_report = df_clean.groupby(['AgeGroup', 'sex_label']).agg(
    PatientCount=('age', 'count'),
    AvgCholesterol=('chol', 'mean'),
    AvgRestingBP=('trestbps', 'mean'),
    DiseaseRate=('target', 'mean')
).reset_index()

summary_report.to_csv('heart_disease_summary_report.csv', index=False)
print('Summary report saved as heart_disease_summary_report.csv')
summary_report

## ✅ Summary of What We Did
1. Loaded the Heart Disease dataset into Pandas
2. Inspected dataset structure (`head`, `info`, `describe`)
3. Identified missing values, duplicates, and invalid records (e.g. 0 cholesterol)
4. Cleaned and transformed the data (removed duplicates, fixed invalid values, added readable labels and age groups)
5. Filtered and sorted data for specific patient segments
6. Grouped and aggregated data to find disease rates by age/gender
7. Visualized key patterns with seaborn/matplotlib
8. Generated written business/analytical insights
9. Exported the cleaned dataset and a summary report as CSV files

This workflow mirrors a typical real-world healthcare analytics pipeline and can be reused for other clinical datasets.